# Liu2024 Source MAT — S-JEPA 500 Hz Sampling-Rate Audit

**Purpose.** Verify, end to end, at what sampling rate the current Liu2024 S-JEPA
pipeline actually preprocesses, builds windows, feeds the encoder, and evaluates —
and then enable a *scientifically clean* native-500 Hz path that does **not** silently
inherit 128 Hz assumptions.

This notebook is a focused **variant** of
`liu2024_source_mat_sjepa_prelocal_augmented_clean.ipynb`. To stay comparable and
avoid code drift, the data-loading, preprocessing, dataset, and model-build cells are
**carried over verbatim** from that notebook; the only behavioural change is that the
window length is defined **in seconds first** and converted to samples at the *effective*
sampling rate, instead of being pinned to a fixed `target_window_samples` count.

### Scope — what this notebook does and does NOT do
- It **does**: audit the sfreq path (static + empirical), enable a config-driven
  native-500 Hz path, run architecture-compatibility diagnostics on the S-JEPA encoder
  at 128 vs 500 Hz, and produce drop-in `config_128hz.json` / `config_500hz.json` plus a
  128-vs-500 results comparison reader.
- It **does not** perform new self-supervised S-JEPA pretraining on Liu data, and it does
  not auto-launch the multi-hour within-subject cross-validation. In this project,
  `pretrained_mode = "from_pretrained"` means *load braindecode's S-JEPA checkpoint and
  fine-tune the new head/spatial layers* — it is transfer + supervised fine-tuning, not
  SSL pretraining on Liu. The heavy CV training stays in the reference notebook and is
  driven here through the two generated config files (see Section 5). If you want a true
  SSL-on-Liu-500Hz pretraining loop, that is a separate deliverable.

### Scientific stance
We are **not** assuming 500 Hz improves accuracy. Downsampling to 128 Hz keeps everything
below ~64 Hz (Nyquist) and the bandpass is 0.5–40 Hz, so for classic sensorimotor-rhythm
MI the discarded band is mostly above the informative range. The audit's job is to make the
trade-off **explicit and measurable**, not to advocate a rate.

# 1. Setup

In [1]:
# Imports. Heavy/optional deps are guarded so the dependency-free parts of the
# sampling-rate audit (Sections 2-3.1, 5.1) still run in a minimal environment.
# This is the one intentional deviation from the reference notebook's unguarded imports.
import copy
import os
import re
import sys
import json
import math
import hashlib
import random
import builtins
import platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from scipy.io import loadmat
from scipy import signal

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as exc:
    HAVE_MPL = False
    print(f"[setup] matplotlib unavailable -> plots will be skipped: {exc}")

try:
    import torch
    from torch.utils.data import Dataset, Subset
    HAVE_TORCH = True
except Exception as exc:
    HAVE_TORCH = False
    Dataset = object  # lets the carried Dataset subclasses still be defined
    print(f"[setup] torch unavailable -> model/architecture sections will be skipped: {exc}")

try:
    import mne
    mne.set_log_level("WARNING")
    HAVE_MNE = True
except Exception as exc:
    HAVE_MNE = False
    print(f"[setup] mne unavailable -> empirical preprocessing audit will be skipped: {exc}")

try:
    from skorch.callbacks import EarlyStopping, EpochScoring
    from skorch.dataset import ValidSplit
    from braindecode import EEGClassifier
    from braindecode.models import SignalJEPA_PreLocal
    HAVE_BRAINDECODE = True
except Exception as exc:
    HAVE_BRAINDECODE = False
    print(f"[setup] braindecode/skorch unavailable -> architecture section will be skipped: {exc}")

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("Runtime Environment:")
print(f"  Python:       {platform.python_version()}")
print(f"  Platform:     {platform.platform()}")
print(f"  numpy:        {np.__version__}")
print(f"  pandas:       {pd.__version__}")
print(f"  HAVE_TORCH:        {HAVE_TORCH}" + (f" (torch {torch.__version__})" if HAVE_TORCH else ""))
print(f"  HAVE_MNE:          {HAVE_MNE}")
print(f"  HAVE_BRAINDECODE:  {HAVE_BRAINDECODE}")
print(f"  HAVE_MPL:          {HAVE_MPL}")


Runtime Environment:
  Python:       3.11.15
  Platform:     Linux-6.14.0-37-generic-x86_64-with-glibc2.41
  numpy:        2.4.3
  pandas:       3.0.1
  HAVE_TORCH:        True (torch 2.10.0+cu128)
  HAVE_MNE:          True
  HAVE_BRAINDECODE:  True
  HAVE_MPL:          True


# 2. Configuration

## 2.1 Liu2024 Channel Defaults

Carried verbatim from the reference notebook. The source MAT is `trials x 33 x samples`;
index 17 (CPz) is the source reference and is dropped, leaving 29 EEG channels. Channel
order and identity must stay identical to the reference for the runs to be comparable.

In [3]:
# Liu2024 source MAT channel conventions.
# Source files are organized as trials x 33 channels x samples:
#   0..29 = EEG-like channels, index 17 = CPz source reference,
#   30..31 = EOG, 32 = marker.
#
# The channel labels below follow the Liu2024 paper / EEGLAB location files.
# This matters for montage-dependent topomaps and any channel-position metadata.
SOURCE_EEG_CHANNEL_INDICES_30 = list(range(30))
SOURCE_REFERENCE_INDEX = 17
SOURCE_EEG_CHANNEL_INDICES_29 = [i for i in SOURCE_EEG_CHANNEL_INDICES_30 if i != SOURCE_REFERENCE_INDEX]

SOURCE_EEG_CHANNEL_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]

# EOG and marker channels available in Liu2024 source MAT files.
SOURCE_EOG_CHANNEL_INDICES = [30, 31]
SOURCE_MARKER_CHANNEL_INDEX = 32


## 2.2 CONFIG

Mirrors the reference `CONFIG` so a run here is comparable to the reference, with three
sampling-rate-relevant changes, each commented inline:

1. **`sfreq_mode`** (new) — single switch that coherently sets resampling. `"native_500"`
   keeps the source 500 Hz; `"resample_128"` reproduces the current pipeline;
   `"custom"` falls back to the raw `resample` / `resample_sfreq` keys.
2. **`target_window_samples = None`** — the reference pinned this to `537`, which silently
   changes the *duration* of the window whenever the sampling rate changes (537 samples is
   4.2 s at 128 Hz but only 1.074 s at 500 Hz). Setting it to `None` makes the window
   **seconds-first**: it is always `round(target_window_s * effective_sfreq)`.
3. **augmentation disabled by default** — `smooth_time_mask`'s `mask_len_samples` is also a
   raw sample count (64 samples = 0.5 s at 128 Hz but 0.128 s at 500 Hz), so leaving it on
   would confound a 128-vs-500 comparison. Re-enable it deliberately, in seconds-aware terms.

In [4]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    # ------------------------------------------------------------------
    # Paths / run identity
    # ------------------------------------------------------------------
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-source-mat-sjepa-pretrain-500hz-audit"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "experiment_name": "sjepa_500hz_audit",
    "config_note": "Sampling-rate audit + seconds-first sfreq-agnostic window definition. "
                   "Carries reference preprocessing/model verbatim; only target_window_samples=None changes behaviour.",
    "config_override_path": None,
    "config_sweep_index": None,

    # ------------------------------------------------------------------
    # Sampling-rate control (NEW). sfreq_mode is the single source of truth and
    # overrides resample/resample_sfreq below in the derived-constants cell.
    #   native_500   -> resample=False, effective sfreq = 500 (the audit focus)
    #   resample_128 -> resample=True,  resample_sfreq = 128 (current pipeline)
    #   custom       -> use resample / resample_sfreq as written
    # ------------------------------------------------------------------
    "sfreq_mode": "native_500",
    "run_tag": "native500",
    "compare_sfreq_modes": ["resample_128", "native_500"],
    "audit_synthetic_if_missing": True,  # use a synthetic 500 Hz trial if no .mat files are found

    # ------------------------------------------------------------------
    # Dataset
    # ------------------------------------------------------------------
    "subjects_to_use": None,
    "exclude_subjects": [],
    "source_unit": "microvolts",
    "final_model_unit": "microvolts",

    # ------------------------------------------------------------------
    # Source-domain preprocessing before creating MNE RawArray
    # ------------------------------------------------------------------
    "demean_mode": "none",
    "baseline_window_s": [0.0, 2.0],
    "detrend_mode": "none",
    "eog_correction": "none",
    "artifact_clip_mode": "none",
    "artifact_clip_abs_value": None,
    "artifact_clip_percentile": 99.5,

    # ------------------------------------------------------------------
    # MNE Raw-level preprocessing (resample/resample_sfreq are managed by sfreq_mode)
    # ------------------------------------------------------------------
    "reference_mode": "average",
    "reference_timing": "before_resample_filter",
    "resample": True,           # overwritten from sfreq_mode in 2.3
    "resample_sfreq": 128,      # overwritten from sfreq_mode in 2.3

    "filter_enabled": True,
    "filter_low": 0.5,
    "filter_high": 40.0,
    "filter_method": "fir",
    "filter_phase": "zero",
    "filter_fir_design": "firwin",
    "filter_l_trans_bandwidth": "auto",
    "filter_h_trans_bandwidth": "auto",
    "filter_iir_params": None,
    "notch_freqs": None,
    "notch_before_bandpass": False,

    # ------------------------------------------------------------------
    # Windowing — SECONDS FIRST. target_window_samples=None => derive from seconds
    # at the effective sampling rate. Do NOT hard-set this unless you intend to pin
    # the sample count and accept a sfreq-dependent change in window DURATION.
    # ------------------------------------------------------------------
    "target_window_s": 4.2,
    "target_window_samples": None,
    "mi_window_start_s": 1.5,

    "reject_bad_trials": False,
    "reject_peak_to_peak_threshold": None,
    "reject_abs_threshold": None,
    "min_trials_per_class_after_reject": None,

    # ------------------------------------------------------------------
    # Fold-safe normalization (fit on training split only)
    # ------------------------------------------------------------------
    "normalization_mode": "none",
    "normalization_eps": 1e-6,

    # ------------------------------------------------------------------
    # Model / downstream strategy
    # ------------------------------------------------------------------
    "model_name": "SignalJEPA_PreLocal",
    "pretrained_mode": "from_pretrained",
    "pretrained_repo_id": "braindecode/signal-jepa_without-chans",
    "strategy": "new",
    "warmup_epochs": 10,

    # ------------------------------------------------------------------
    # Evaluation protocol (within-subject, identical to reference)
    # ------------------------------------------------------------------
    "evaluation_mode": "stratified_kfold",
    "cv_folds": 5,
    "n_repeats": 10,
    "test_size": 0.4,
    "split_random_state": 2026,
    "assert_balanced_folds": True,

    # ------------------------------------------------------------------
    # Training hyperparameters (used only if you run full CV via the reference pipeline)
    # ------------------------------------------------------------------
    "batch_size": 4,
    "n_epochs": 5000,
    "early_stopping_patience": 50,
    "val_split": 0.2,
    "learning_rate": 0.0003,
    "optimizer_name": "adam",
    "weight_decay": 0.0,
    "gradient_clip_norm": None,
    "checkpoint_metric": "valid_loss",
    "label_smoothing": 0.0,
    "prediction_balance_loss_weight": 1.0,

    # ------------------------------------------------------------------
    # Augmentation OFF by default for a clean sfreq comparison (see 2.2, note 3).
    # ------------------------------------------------------------------
    "augmentation": {
        "enabled": False,
        "name": "none",
        "random_state": 2026,
        "transforms": [],
    },

    # ------------------------------------------------------------------
    # Reproducibility
    # ------------------------------------------------------------------
    "seed": 2026,
    "set_seed": True,
    "cv_random_state": 2026,
    "val_split_random_state": 2026,

    # ------------------------------------------------------------------
    # Diagnostics (heavy spatial-conv interpretability left in the reference notebook)
    # ------------------------------------------------------------------
    "extract_spatial_conv_weights": False,
    "collapse_threshold": 0.9,
    "log_probability_diagnostics": True,
}


## 2.3 Derived constants — seconds-first, sampling-rate agnostic

`resolve_sampling_and_window()` is the heart of the fix. It turns `sfreq_mode` into a
concrete (`resample`, `effective_sfreq`) pair, then derives every window quantity **from
seconds**, and reports the implied duration of any legacy fixed sample count so a silent
change can never go unnoticed. The resolved values are written back into `CONFIG` and into
module-level globals (`EFFECTIVE_SFREQ`, `WINDOW_SAMPLES`, ...) with the same names the
carried reference cells expect.

In [5]:
# --- Fixed Liu2024 source-MAT constants (carried from the reference notebook) ---
LIU_SOURCE_SFREQ = 500
LIU_EXPECTED_TRIALS_PER_SUBJECT = 40
LIU_EXPECTED_SOURCE_CHANNELS = 33
LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL = 4000

EEG_CHANNEL_INDICES = SOURCE_EEG_CHANNEL_INDICES_29
EEG_CHANNEL_NAMES = [
    name for idx, name in enumerate(SOURCE_EEG_CHANNEL_NAMES_30)
    if idx != SOURCE_REFERENCE_INDEX
]
SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
TARGET_N_CLASSES = 2


def resolve_sfreq_mode(config):
    """Turn sfreq_mode into a concrete (resample, resample_sfreq, effective_sfreq) triple."""
    mode = str(config.get("sfreq_mode", "custom")).lower()
    source_sfreq = float(LIU_SOURCE_SFREQ)
    if mode == "native_500":
        resample, resample_sfreq = False, source_sfreq
    elif mode == "resample_128":
        resample, resample_sfreq = True, 128.0
    elif mode == "custom":
        resample = bool(config.get("resample", True))
        resample_sfreq = float(config.get("resample_sfreq", 128.0))
    else:
        raise ValueError(f"Unsupported sfreq_mode={config.get('sfreq_mode')}")
    effective_sfreq = float(resample_sfreq) if resample else source_sfreq
    return {
        "sfreq_mode": mode,
        "source_sfreq": source_sfreq,
        "resample": bool(resample),
        "resample_sfreq": float(resample_sfreq),
        "effective_sfreq": float(effective_sfreq),
    }


def resolve_sampling_and_window(config):
    """Derive every window quantity from SECONDS at the effective sampling rate.

    Returns a fully explicit dict. Never relies on a hard-coded sample count: if a legacy
    target_window_samples is present, it is reported alongside its implied duration and the
    delta versus the seconds-first definition, but the seconds-first value governs.
    """
    sr = resolve_sfreq_mode(config)
    eff = sr["effective_sfreq"]
    src = sr["source_sfreq"]

    window_start_s = float(config["mi_window_start_s"])
    window_len_s = float(config["target_window_s"])

    window_samples = int(round(window_len_s * eff))
    start_sample = int(round(window_start_s * eff))
    stop_sample = start_sample + window_samples
    duration_s = window_samples / eff

    source_samples_per_trial = int(LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL)
    resampled_samples_per_trial = int(round(source_samples_per_trial * eff / src))

    legacy = config.get("target_window_samples", None)
    legacy_block = None
    if legacy is not None:
        legacy = int(legacy)
        legacy_block = {
            "legacy_target_window_samples": legacy,
            "legacy_implied_duration_s": round(legacy / eff, 6),
            "seconds_first_window_samples": window_samples,
            "duration_delta_s": round((window_samples - legacy) / eff, 6),
            "note": "Legacy fixed sample count IGNORED; seconds-first value governs.",
        }

    info = dict(sr)
    info.update({
        "window_start_s": window_start_s,
        "window_len_s_requested": window_len_s,
        "window_samples": int(window_samples),
        "start_sample": int(start_sample),
        "stop_sample": int(stop_sample),
        "effective_window_duration_s": round(duration_s, 6),
        "source_samples_per_trial": source_samples_per_trial,
        "resampled_samples_per_trial": resampled_samples_per_trial,
        "window_fits_in_trial": bool(stop_sample <= resampled_samples_per_trial),
        "legacy_fixed_window_check": legacy_block,
    })
    return info


# Resolve for the ACTIVE config and write back the canonical globals/keys.
WINDOW_INFO = resolve_sampling_and_window(CONFIG)
if not WINDOW_INFO["window_fits_in_trial"]:
    raise ValueError(
        f"Window [{WINDOW_INFO['start_sample']}:{WINDOW_INFO['stop_sample']}] exceeds the "
        f"resampled trial length {WINDOW_INFO['resampled_samples_per_trial']} at "
        f"effective_sfreq={WINDOW_INFO['effective_sfreq']}."
    )

CONFIG["resample"] = WINDOW_INFO["resample"]
CONFIG["resample_sfreq"] = WINDOW_INFO["resample_sfreq"]
CONFIG["effective_sfreq"] = WINDOW_INFO["effective_sfreq"]
CONFIG["sfreq"] = WINDOW_INFO["effective_sfreq"]  # compat with carried cells
# target_window_samples stays None so the carried preprocessing derives from seconds.

EFFECTIVE_SFREQ = float(WINDOW_INFO["effective_sfreq"])
WINDOW_SAMPLES = int(WINDOW_INFO["window_samples"])
MI_WINDOW_START_SAMPLE = int(WINDOW_INFO["start_sample"])
MI_WINDOW_STOP_SAMPLE = int(WINDOW_INFO["stop_sample"])
TARGET_TRIAL_DURATION_S = float(WINDOW_INFO["effective_window_duration_s"])

# Artifact-key groups (mirror reference + sfreq audit keys).
PREPROCESSING_KEYS = [
    "sfreq_mode", "source_unit", "final_model_unit",
    "demean_mode", "baseline_window_s", "detrend_mode", "eog_correction",
    "artifact_clip_mode", "artifact_clip_abs_value", "artifact_clip_percentile",
    "reference_mode", "reference_timing", "resample", "resample_sfreq", "effective_sfreq",
    "filter_enabled", "filter_low", "filter_high", "filter_method", "filter_phase",
    "filter_fir_design", "filter_l_trans_bandwidth", "filter_h_trans_bandwidth", "filter_iir_params",
    "notch_freqs", "notch_before_bandpass",
    "mi_window_start_s", "target_window_s", "target_window_samples",
    "reject_bad_trials", "reject_peak_to_peak_threshold", "reject_abs_threshold",
    "normalization_mode", "normalization_eps",
]
EVALUATION_KEYS = [
    "evaluation_mode", "cv_folds", "n_repeats", "test_size",
    "cv_random_state", "split_random_state", "val_split_random_state",
]

def summarize_selected_config(keys):
    return {k: CONFIG.get(k) for k in keys}

PREPROCESSING_CONFIG = summarize_selected_config(PREPROCESSING_KEYS)
EVALUATION_CONFIG = summarize_selected_config(EVALUATION_KEYS)

print("Resolved sampling / window settings (ACTIVE config):")
for k, v in WINDOW_INFO.items():
    if k == "legacy_fixed_window_check":
        continue
    print(f"  {k:32s}: {v}")
print(f"  channels                        : {len(EEG_CHANNEL_NAMES)} -> {EEG_CHANNEL_NAMES}")


Resolved sampling / window settings (ACTIVE config):
  sfreq_mode                      : native_500
  source_sfreq                    : 500.0
  resample                        : False
  resample_sfreq                  : 500.0
  effective_sfreq                 : 500.0
  window_start_s                  : 1.5
  window_len_s_requested          : 4.2
  window_samples                  : 2100
  start_sample                    : 750
  stop_sample                     : 2850
  effective_window_duration_s     : 4.2
  source_samples_per_trial        : 4000
  resampled_samples_per_trial     : 4000
  window_fits_in_trial            : True
  channels                        : 29 -> ['Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8', 'FCz', 'FC3', 'FC4', 'FT7', 'FT8', 'Cz', 'C3', 'C4', 'T3', 'T4', 'CP3', 'CP4', 'TP7', 'TP8', 'Pz', 'P3', 'P4', 'T5', 'T6', 'Oz', 'O1', 'O2']


## 2.4 Artifact creation and logging init

Same run-folder + tee-to-log pattern as the reference, except the run-folder name encodes
the sampling-rate so 128 Hz and 500 Hz runs never collide:
`{timestamp}_{run_tag}_sr{effective_sfreq}_{config_hash}`.

In [6]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    tag = str(CONFIG.get("run_tag", "run"))
    sr = int(round(EFFECTIVE_SFREQ))
    return f"{timestamp}_{tag}_sr{sr}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")

def _safe_write_text(stream, text):
    try:
        stream.write(text)
        return
    except UnicodeEncodeError:
        pass
    encoding = getattr(stream, "encoding", None) or "utf-8"
    safe_text = text.encode(encoding, errors="replace").decode(encoding, errors="replace")
    stream.write(safe_text)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)
    message = sep.join(str(arg) for arg in args)
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            _safe_write_text(sys.stdout, text)
            if flush:
                sys.stdout.flush()
        else:
            _safe_write_text(file, text)
            if flush and hasattr(file, "flush"):
                file.flush()

    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _safe_write_text(_LOG_FILE_HANDLE, blanks)
    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _safe_write_text(_LOG_FILE_HANDLE, stamped + end)
    else:
        _write_target(end)
        _safe_write_text(_LOG_FILE_HANDLE, end)
    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)

print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Config:     {config_path}")


[2026-06-14 13:45:53] Run ID:     20260614_1345_native500_sr500_efdd1f75
[2026-06-14 13:45:53] Artifacts:  /home/vegorov/Repos/eeg_jepa_research/artifacts/liu2024-source-mat-sjepa-pretrain-500hz-audit/20260614_1345_native500_sr500_efdd1f75
[2026-06-14 13:45:53] Config:     /home/vegorov/Repos/eeg_jepa_research/artifacts/liu2024-source-mat-sjepa-pretrain-500hz-audit/20260614_1345_native500_sr500_efdd1f75/config.json


## 2.5 Reproducibility

In [7]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

BASE_SEED = int(CONFIG["seed"])
if CONFIG["set_seed"]:
    seed_everything(BASE_SEED)
    print(f"Seed initialized: {BASE_SEED}")


[2026-06-14 13:45:53] Using device: cpu
[2026-06-14 13:45:54] Seed initialized: 2026


# 3. Sampling-Rate Audit

The core deliverable. Section 3.1 audits the path purely from configuration (no data, no
heavy deps). Sections 3.2-3.4 carry the real machinery and confirm the audit empirically on
an actual (or clearly-labelled synthetic) trial.

## 3.1 Static sampling-rate audit (configuration-only)

Answers, without touching data: where the source starts (500 Hz), whether/where it is
resampled, the effective sampling rate used downstream, and the exact window
seconds-to-samples conversion — computed for **every** mode in `compare_sfreq_modes` so
128 Hz and 500 Hz sit side by side.

In [8]:
def static_sfreq_audit_row(sfreq_mode):
    cfg = copy.deepcopy(CONFIG)
    cfg["sfreq_mode"] = sfreq_mode
    info = resolve_sampling_and_window(cfg)
    row = {
        "sfreq_mode": sfreq_mode,
        "source_sfreq_hz": info["source_sfreq"],
        "resample": info["resample"],
        "effective_sfreq_hz": info["effective_sfreq"],
        "source_samples_per_trial": info["source_samples_per_trial"],
        "resampled_samples_per_trial": info["resampled_samples_per_trial"],
        "window_start_s": info["window_start_s"],
        "window_len_s": info["window_len_s_requested"],
        "window_samples": info["window_samples"],
        "crop_start_sample": info["start_sample"],
        "crop_stop_sample": info["stop_sample"],
        "window_duration_s": info["effective_window_duration_s"],
        "window_fits_in_trial": info["window_fits_in_trial"],
    }
    return row, info

audit_rows = []
audit_full = {}
for mode in CONFIG.get("compare_sfreq_modes", ["resample_128", "native_500"]):
    row, info = static_sfreq_audit_row(mode)
    audit_rows.append(row)
    audit_full[mode] = info

static_audit_df = pd.DataFrame(audit_rows)
print("=" * 78)
print("STATIC SAMPLING-RATE AUDIT")
print("=" * 78)
print(f"  Source MAT sampling rate (fixed): {LIU_SOURCE_SFREQ} Hz")
print(f"  Source samples/trial (fixed):     {LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL} "
      f"(= {LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL / LIU_SOURCE_SFREQ:.3f} s/trial)")
print(f"  ACTIVE sfreq_mode:                {CONFIG['sfreq_mode']} "
      f"(effective {EFFECTIVE_SFREQ:g} Hz)")
print()
try:
    from IPython.display import display
    display(static_audit_df)
except Exception:
    print(static_audit_df.to_string(index=False))

# Surface the silent-pinning hazard explicitly for the professor-facing write-up.
print("\nWindow-duration sanity across rates (seconds-first window of "
      f"{CONFIG['target_window_s']} s):")
for mode, info in audit_full.items():
    print(f"  {mode:14s}: {info['window_samples']:5d} samples @ {info['effective_sfreq']:g} Hz "
          f"= {info['effective_window_duration_s']:.4f} s")
print("\nReference notebook pinned target_window_samples=537. Implied durations of THAT "
      "fixed count, had it been kept:")
for mode, info in audit_full.items():
    eff = info["effective_sfreq"]
    print(f"  {mode:14s}: 537 samples @ {eff:g} Hz = {537/eff:.4f} s  "
          f"(seconds-first here uses {info['window_samples']} samples = {info['effective_window_duration_s']:.4f} s)")

# Note the augmentation sample-count hazard too (off by default here).
print("\nNote: smooth_time_mask mask_len_samples is also a raw sample count. "
      "64 samples = 0.500 s @128 Hz but 0.128 s @500 Hz. Augmentation is disabled in this "
      "audit to keep the comparison clean.")

static_audit_path = ARTIFACT_DIR / "sfreq_audit_static.csv"
static_audit_df.to_csv(static_audit_path, index=False)
with open(ARTIFACT_DIR / "sfreq_audit_static.json", "w") as f:
    json.dump({"active_sfreq_mode": CONFIG["sfreq_mode"], "modes": audit_full}, f, indent=2, default=str)
print(f"\nStatic audit saved to: {static_audit_path}")


[2026-06-14 13:45:54] ==============================================================================
[2026-06-14 13:45:54] STATIC SAMPLING-RATE AUDIT
[2026-06-14 13:45:54] ==============================================================================
[2026-06-14 13:45:54]   Source MAT sampling rate (fixed): 500 Hz
[2026-06-14 13:45:54]   Source samples/trial (fixed):     4000 (= 8.000 s/trial)
[2026-06-14 13:45:54]   ACTIVE sfreq_mode:                native_500 (effective 500 Hz)



,sfreq_mode,source_sfreq_hz,resample,effective_sfreq_hz,source_samples_per_trial,resampled_samples_per_trial,window_start_s,window_len_s,window_samples,crop_start_sample,crop_stop_sample,window_duration_s,window_fits_in_trial
0,resample_128,500.0,True,128.0,4000,1024,1.5,4.2,538,192,730,4.203125,True
1,native_500,500.0,False,500.0,4000,4000,1.5,4.2,2100,750,2850,4.200000,True



[2026-06-14 13:45:54] Window-duration sanity across rates (seconds-first window of 4.2 s):
[2026-06-14 13:45:54]   resample_128  :   538 samples @ 128 Hz = 4.2031 s
[2026-06-14 13:45:54]   native_500    :  2100 samples @ 500 Hz = 4.2000 s

[2026-06-14 13:45:54] Reference notebook pinned target_window_samples=537. Implied durations of THAT fixed count, had it been kept:
[2026-06-14 13:45:54]   resample_128  : 537 samples @ 128 Hz = 4.1953 s  (seconds-first here uses 538 samples = 4.2031 s)
[2026-06-14 13:45:54]   native_500    : 537 samples @ 500 Hz = 1.0740 s  (seconds-first here uses 2100 samples = 4.2000 s)

[2026-06-14 13:45:54] Note: smooth_time_mask mask_len_samples is also a raw sample count. 64 samples = 0.500 s @128 Hz but 0.128 s @500 Hz. Augmentation is disabled in this audit to keep the comparison clean.

[2026-06-14 13:45:54] Static audit saved to: /home/vegorov/Repos/eeg_jepa_research/artifacts/liu2024-source-mat-sjepa-pretrain-500hz-audit/20260614_1345_native500_sr500_ef

## 3.2 Carried machinery — data loading, preprocessing, datasets

Verbatim from the reference notebook (single source of truth). The preprocessing already
derives the window from `target_window_s` when `target_window_samples is None`, so with our
`None` default it is fully seconds-first and sampling-rate agnostic. `apply_resample` reads
`CONFIG["resample"]` / `CONFIG["resample_sfreq"]`, which Section 2.3 set from `sfreq_mode`.

In [9]:
def make_liu_info(sfreq):
    info = mne.create_info(
        ch_names=EEG_CHANNEL_NAMES,
        sfreq=float(sfreq),
        ch_types=["eeg"] * len(EEG_CHANNEL_NAMES),  # type: ignore
    )
    montage = mne.channels.make_standard_montage("standard_1020")
    info.set_montage(montage, match_case=False, on_missing="ignore")
    return info

def labels_to_zero_based(labels):
    labels = np.asarray(labels).astype(int).ravel()
    unique = set(np.unique(labels).tolist())
    if unique.issubset({1, 2}):
        return labels - 1
    if unique.issubset({0, 1}):
        return labels
    raise ValueError(f"Unexpected labels: {sorted(unique)}")

def source_values_to_mne_volts(data, config=None):
    config = CONFIG if config is None else config
    arr = np.asarray(data, dtype=np.float64)
    unit = str(config.get("source_unit", "microvolts")).lower()
    if unit in ("uv", "microvolt", "microvolts"):
        return arr * 1e-6
    if unit in ("mv", "millivolt", "millivolts"):
        return arr * 1e-3
    if unit in ("v", "volt", "volts"):
        return arr
    raise ValueError(f"Unsupported source_unit={config.get('source_unit')}")

def mne_volts_to_model_unit(data, config=None):
    config = CONFIG if config is None else config
    arr = np.asarray(data, dtype=np.float64)
    unit = str(config.get("final_model_unit", "microvolts")).lower()
    if unit in ("uv", "microvolt", "microvolts"):
        return arr * 1e6
    if unit in ("v", "volt", "volts"):
        return arr
    raise ValueError(f"Unsupported final_model_unit={config.get('final_model_unit')}")

def _none_like(value):
    return value is None or str(value).lower() in ("none", "off", "false", "")

def build_preprocessing_pipeline(config):
    """Return a readable pipeline plan.

    The pipeline is represented as a list of dictionaries rather than hidden global logic.
    Every row is logged and saved in the run metadata through the preprocessing step list.
    """
    pipeline = []

    # Fixed source structure.
    pipeline.append({
        "stage": "source",
        "name": "select_eeg_channels",
        "description": "select Liu EEG channels, drop CPz reference, EOG, and marker before model input",
        "enabled": True,
    })

    pipeline.append({
        "stage": "source",
        "name": "demean",
        "mode": config.get("demean_mode", "none"),
        "baseline_window_s": config.get("baseline_window_s"),
        "enabled": not _none_like(config.get("demean_mode", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "detrend",
        "mode": config.get("detrend_mode", "none"),
        "enabled": not _none_like(config.get("detrend_mode", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "eog_correction",
        "mode": config.get("eog_correction", "none"),
        "enabled": not _none_like(config.get("eog_correction", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "artifact_clipping",
        "mode": config.get("artifact_clip_mode", "none"),
        "abs_value": config.get("artifact_clip_abs_value"),
        "percentile": config.get("artifact_clip_percentile"),
        "enabled": not _none_like(config.get("artifact_clip_mode", "none")),
    })

    reference_timing = str(config.get("reference_timing", "before_resample_filter")).lower()
    if reference_timing == "before_resample_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    if bool(config.get("notch_before_bandpass", False)):
        pipeline.append({"stage": "mne_raw", "name": "notch_filter", "timing": "before_bandpass", "freqs": config.get("notch_freqs"), "enabled": not _none_like(config.get("notch_freqs"))})

    pipeline.append({"stage": "mne_raw", "name": "resample", "sfreq": config.get("resample_sfreq"), "enabled": bool(config.get("resample", True))})

    if reference_timing == "after_resample_before_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    pipeline.append({
        "stage": "mne_raw",
        "name": "bandpass_filter",
        "enabled": bool(config.get("filter_enabled", True)),
        "l_freq": config.get("filter_low"),
        "h_freq": config.get("filter_high"),
        "method": config.get("filter_method"),
        "phase": config.get("filter_phase"),
        "fir_design": config.get("filter_fir_design"),
        "iir_params": config.get("filter_iir_params"),
    })

    if reference_timing == "after_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    if not bool(config.get("notch_before_bandpass", False)):
        pipeline.append({"stage": "mne_raw", "name": "notch_filter", "timing": "after_bandpass", "freqs": config.get("notch_freqs"), "enabled": not _none_like(config.get("notch_freqs"))})

    pipeline.append({
        "stage": "window",
        "name": "crop_fixed_mi_window",
        "start_s": config.get("mi_window_start_s"),
        "target_window_samples": config.get("target_window_samples"),
        "enabled": True,
    })

    pipeline.append({
        "stage": "window",
        "name": "bad_trial_rejection",
        "enabled": bool(config.get("reject_bad_trials", False)),
        "peak_to_peak_threshold": config.get("reject_peak_to_peak_threshold"),
        "abs_threshold": config.get("reject_abs_threshold"),
    })

    pipeline.append({
        "stage": "split",
        "name": "fold_safe_normalization",
        "mode": config.get("normalization_mode", "none"),
        "enabled": not _none_like(config.get("normalization_mode", "none")),
    })

    return pipeline

def describe_pipeline(pipeline):
    lines = []
    for step in pipeline:
        status = "ON" if step.get("enabled", False) else "off"
        parts = [f"[{status}] {step.get('stage')}::{step.get('name')}"]
        for key, value in step.items():
            if key not in ("stage", "name", "description", "enabled") and value is not None:
                parts.append(f"{key}={value}")
        if step.get("description"):
            parts.append(f"- {step['description']}")
        lines.append(" | ".join(parts))
    return lines

PREPROCESSING_PIPELINE = build_preprocessing_pipeline(CONFIG)
print("Configured preprocessing pipeline:")
for line in describe_pipeline(PREPROCESSING_PIPELINE):
    print("  - " + line)

def apply_source_demean(X_eeg, subject_id, config, steps):
    mode = str(config.get("demean_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)

    if mode in ("none", "off", "false"):
        steps.append("source demean skipped")
        return X

    if mode == "trial_mean":
        steps.append("source demean: subtract trial/channel mean over time")
        return X - X.mean(axis=-1, keepdims=True)

    if mode == "baseline_window_mean":
        baseline = config.get("baseline_window_s", [0.0, 2.0])
        if baseline is None or len(baseline) != 2:
            raise ValueError("baseline_window_s must be [start_s, stop_s] for baseline_window_mean.")
        start_s, stop_s = float(baseline[0]), float(baseline[1])
        start = int(round(start_s * LIU_SOURCE_SFREQ))
        stop = int(round(stop_s * LIU_SOURCE_SFREQ))
        if start < 0 or stop <= start or stop > X.shape[-1]:
            raise ValueError(f"Subject {subject_id}: invalid baseline_window_s={baseline} for source length {X.shape[-1]}")
        steps.append(f"source demean: subtract baseline mean {baseline}s")
        return X - X[:, :, start:stop].mean(axis=-1, keepdims=True)

    raise ValueError(f"Unsupported demean_mode={config.get('demean_mode')}")

def apply_source_detrend(X_eeg, subject_id, config, steps):
    mode = str(config.get("detrend_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)
    if mode in ("none", "off", "false"):
        steps.append("source detrend skipped")
        return X
    if mode == "constant":
        steps.append("source detrend: scipy.signal.detrend(type='constant')")
        return signal.detrend(X, axis=-1, type="constant")
    if mode == "linear":
        steps.append("source detrend: scipy.signal.detrend(type='linear')")
        return signal.detrend(X, axis=-1, type="linear")
    raise ValueError(f"Unsupported detrend_mode={config.get('detrend_mode')}")

def apply_eog_correction(X_eeg, rawdata, subject_id, config, steps):
    mode = str(config.get("eog_correction", "none")).lower()
    if mode in ("none", "off", "false"):
        steps.append("EOG correction skipped")
        return X_eeg

    if mode != "linear_regression":
        raise ValueError(
            "Only eog_correction='linear_regression' is implemented in this source-MAT notebook. "
            "ICA is intentionally not included because the source pipeline drops EOG before model input and "
            "the dataset has only 40 trials per subject."
        )

    if rawdata.shape[1] <= max(SOURCE_EOG_CHANNEL_INDICES):
        raise ValueError(f"Subject {subject_id}: rawdata does not contain expected EOG channels.")

    X = np.asarray(X_eeg, dtype=np.float64)
    eog = np.asarray(rawdata[:, SOURCE_EOG_CHANNEL_INDICES, :], dtype=np.float64)

    n_trials, n_chans, n_samples = X.shape
    eog_2d = eog.transpose(0, 2, 1).reshape(-1, len(SOURCE_EOG_CHANNEL_INDICES))
    eeg_2d = X.transpose(0, 2, 1).reshape(-1, n_chans)

    design = np.column_stack([np.ones(eog_2d.shape[0]), eog_2d])
    beta, *_ = np.linalg.lstsq(design, eeg_2d, rcond=None)
    eog_contribution = design[:, 1:] @ beta[1:, :]
    corrected = eeg_2d - eog_contribution
    corrected = corrected.reshape(n_trials, n_samples, n_chans).transpose(0, 2, 1)

    steps.append("EOG correction: linear regression using HEOG/VEOG before dropping EOG")
    return corrected

def apply_source_artifact_clipping(X_eeg, subject_id, config, steps):
    mode = str(config.get("artifact_clip_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)

    if mode in ("none", "off", "false"):
        steps.append("source artifact clipping skipped")
        return X, {"artifact_clip_applied": False, "artifact_clip_threshold": None}

    if mode == "absolute":
        threshold = config.get("artifact_clip_abs_value")
        if threshold is None:
            raise ValueError("artifact_clip_abs_value must be set when artifact_clip_mode='absolute'.")
        threshold = float(threshold)
        clipped = np.clip(X, -threshold, threshold)
        changed = int(np.sum(clipped != X))
        steps.append(f"source artifact clipping: absolute ±{threshold:g} {config.get('source_unit')} | changed_values={changed}")
        return clipped, {
            "artifact_clip_applied": True,
            "artifact_clip_mode": mode,
            "artifact_clip_threshold": threshold,
            "artifact_clip_changed_values": changed,
        }

    if mode == "percentile":
        pct = float(config.get("artifact_clip_percentile", 99.5))
        threshold = float(np.nanpercentile(np.abs(X), pct))
        clipped = np.clip(X, -threshold, threshold)
        changed = int(np.sum(clipped != X))
        steps.append(f"source artifact clipping: percentile {pct:g}% -> ±{threshold:.4g} {config.get('source_unit')} | changed_values={changed}")
        return clipped, {
            "artifact_clip_applied": True,
            "artifact_clip_mode": mode,
            "artifact_clip_percentile": pct,
            "artifact_clip_threshold": threshold,
            "artifact_clip_changed_values": changed,
        }

    raise ValueError(f"Unsupported artifact_clip_mode={config.get('artifact_clip_mode')}")

def apply_reference(raw, config, steps, timing_label):
    mode = str(config.get("reference_mode", "average")).lower()
    if mode in ("none", "off", "false"):
        steps.append(f"reference skipped at {timing_label}")
        return raw
    if mode == "average":
        raw.set_eeg_reference("average", projection=False, verbose=False)
        steps.append(f"average reference at {timing_label}")
        return raw
    raise ValueError(f"Unsupported reference_mode={config.get('reference_mode')}")

def apply_notch(raw, config, steps, timing_label):
    freqs = config.get("notch_freqs", None)
    if freqs is None or freqs == []:
        return raw
    raw.notch_filter(freqs=freqs, verbose=False)
    steps.append(f"notch filter {freqs} Hz at {timing_label}")
    return raw

def apply_resample(raw, config, steps):
    if bool(config.get("resample", True)):
        target = float(config.get("resample_sfreq", 128))
        raw.resample(target, verbose=False)
        steps.append(f"resample to {target:g} Hz")
    else:
        steps.append("resample skipped; kept source 500 Hz")
    return raw

def _float_or_none_or_auto(value):
    if value is None:
        return None
    if isinstance(value, str) and value.lower() == "auto":
        return "auto"
    return float(value)

def apply_bandpass(raw, config, steps):
    if not bool(config.get("filter_enabled", True)):
        steps.append("bandpass skipped")
        return raw

    l_freq = config.get("filter_low", None)
    h_freq = config.get("filter_high", None)
    l_freq = None if l_freq is None else float(l_freq)
    h_freq = None if h_freq is None else float(h_freq)

    method = str(config.get("filter_method", "fir")).lower()
    if method == "iir":
        iir_params = config.get("filter_iir_params", None)
        if iir_params is None:
            iir_params = {"order": 2, "ftype": "butter"}
        raw.filter(l_freq=l_freq, h_freq=h_freq, method="iir", iir_params=iir_params, verbose=False)
        steps.append(f"IIR bandpass {l_freq}–{h_freq} Hz | params={iir_params}")
    elif method == "fir":
        raw.filter(
            l_freq=l_freq,
            h_freq=h_freq,
            method="fir",
            phase=config.get("filter_phase", "zero"),
            fir_design=config.get("filter_fir_design", "firwin"),
            l_trans_bandwidth=_float_or_none_or_auto(config.get("filter_l_trans_bandwidth", "auto")),
            h_trans_bandwidth=_float_or_none_or_auto(config.get("filter_h_trans_bandwidth", "auto")),
            verbose=False,
        )
        steps.append(
            f"FIR bandpass {l_freq}–{h_freq} Hz | phase={config.get('filter_phase')} | "
            f"fir_design={config.get('filter_fir_design')}"
        )
    else:
        raise ValueError(f"Unsupported filter_method={config.get('filter_method')}")
    return raw

def apply_mne_raw_pipeline(raw, config, steps):
    reference_timing = str(config.get("reference_timing", "before_resample_filter")).lower()

    if reference_timing == "before_resample_filter":
        raw = apply_reference(raw, config, steps, "before resample/filter")

    if bool(config.get("notch_before_bandpass", False)):
        raw = apply_notch(raw, config, steps, "before resample/filter")

    raw = apply_resample(raw, config, steps)

    if reference_timing == "after_resample_before_filter":
        raw = apply_reference(raw, config, steps, "after resample before filter")

    raw = apply_bandpass(raw, config, steps)

    if reference_timing == "after_filter":
        raw = apply_reference(raw, config, steps, "after filter")

    if not bool(config.get("notch_before_bandpass", False)):
        raw = apply_notch(raw, config, steps, "after bandpass")

    return raw

def maybe_reject_bad_trials(X_win, y, subject_id, config, steps):
    stats = {
        "reject_bad_trials": bool(config.get("reject_bad_trials", False)),
        "n_trials_before_reject": int(len(y)),
        "n_trials_after_reject": int(len(y)),
        "n_rejected_trials": 0,
        "rejection_skipped": False,
    }

    if not bool(config.get("reject_bad_trials", False)):
        steps.append("bad-trial rejection skipped")
        return X_win, y, stats

    keep = np.ones(len(y), dtype=bool)

    ptp_threshold = config.get("reject_peak_to_peak_threshold", None)
    if ptp_threshold is not None:
        ptp = np.ptp(X_win, axis=-1).max(axis=1)
        keep &= ptp <= float(ptp_threshold)
        stats["reject_peak_to_peak_threshold"] = float(ptp_threshold)
        stats["max_trial_peak_to_peak"] = float(np.max(ptp))

    abs_threshold = config.get("reject_abs_threshold", None)
    if abs_threshold is not None:
        max_abs = np.max(np.abs(X_win), axis=(1, 2))
        keep &= max_abs <= float(abs_threshold)
        stats["reject_abs_threshold"] = float(abs_threshold)
        stats["max_trial_abs"] = float(np.max(max_abs))

    proposed_y = y[keep]
    min_required = config.get("min_trials_per_class_after_reject", None)
    if min_required is None:
        if config.get("evaluation_mode") == "liu2024_repeated_60_40":
            min_required = 12
        else:
            min_required = int(config.get("cv_folds", 5))
    proposed_counts = np.bincount(proposed_y, minlength=TARGET_N_CLASSES)

    if len(proposed_y) == 0 or proposed_counts.min() < int(min_required):
        steps.append(
            "bad-trial rejection skipped because it would leave too few samples "
            f"per class: proposed_counts={proposed_counts.tolist()}, min_required={min_required}"
        )
        stats["rejection_skipped"] = True
        stats["proposed_class_counts_after_reject"] = proposed_counts.tolist()
        return X_win, y, stats

    X_new = X_win[keep]
    y_new = proposed_y
    stats["n_trials_after_reject"] = int(len(y_new))
    stats["n_rejected_trials"] = int(np.sum(~keep))
    stats["class_counts_after_reject"] = np.bincount(y_new, minlength=TARGET_N_CLASSES).tolist()
    steps.append(
        f"bad-trial rejection applied: rejected={stats['n_rejected_trials']} / {stats['n_trials_before_reject']} | "
        f"class_counts={stats['class_counts_after_reject']}"
    )
    return X_new, y_new, stats

def preprocess_subject_configurable(rawdata, labels, subject_id, config=None):
    """Apply the configured preprocessing pipeline to one Liu2024 source subject.

    Order:
      1. source-domain operations: channel selection, mean removal, detrending, EOG regression, clipping
      2. MNE RawArray operations: reference, notch, resample, bandpass
      3. window crop and optional trial rejection
      4. fold-safe normalization later inside training split code
    """
    config = CONFIG if config is None else config

    if rawdata.ndim != 3:
        raise ValueError(f"Subject {subject_id}: expected 3D rawdata, got {rawdata.shape}")

    n_trials = rawdata.shape[0]
    steps = describe_pipeline(build_preprocessing_pipeline(config))
    runtime_steps = []
    preprocessing_stats = {}

    X_eeg = rawdata[:, EEG_CHANNEL_INDICES, :].astype(np.float64)
    runtime_steps.append("select 29 EEG channels; drop CPz source reference, EOG, and marker")

    X_eeg = apply_source_demean(X_eeg, subject_id, config, runtime_steps)
    X_eeg = apply_source_detrend(X_eeg, subject_id, config, runtime_steps)
    X_eeg = apply_eog_correction(X_eeg, rawdata, subject_id, config, runtime_steps)
    X_eeg, clip_stats = apply_source_artifact_clipping(X_eeg, subject_id, config, runtime_steps)
    preprocessing_stats.update(clip_stats)

    X_eeg_volts = source_values_to_mne_volts(X_eeg, config)
    runtime_steps.append(f"convert source {config.get('source_unit')} to MNE volts")

    continuous = X_eeg_volts.transpose(1, 0, 2).reshape(len(EEG_CHANNEL_INDICES), -1)
    info = make_liu_info(LIU_SOURCE_SFREQ)
    raw = mne.io.RawArray(continuous, info, verbose=False)

    raw = apply_mne_raw_pipeline(raw, config, runtime_steps)

    effective_sfreq = float(raw.info["sfreq"])
    data = mne_volts_to_model_unit(raw.get_data(), config)
    runtime_steps.append(f"convert MNE volts to model {config.get('final_model_unit')}")

    expected_samples_per_trial = int(round(rawdata.shape[2] * effective_sfreq / float(LIU_SOURCE_SFREQ)))
    total_expected = n_trials * expected_samples_per_trial
    if data.shape[1] != total_expected:  # type: ignore
        n_full = data.shape[1] // n_trials  # type: ignore
        expected_samples_per_trial = n_full
        data = data[:, :n_trials * expected_samples_per_trial]  # type: ignore
        runtime_steps.append(f"trim continuous samples to full trials: {expected_samples_per_trial} samples/trial")

    X_rs = data.reshape(len(EEG_CHANNEL_INDICES), n_trials, expected_samples_per_trial).transpose(1, 0, 2)  # type: ignore

    start_sample = int(round(float(config["mi_window_start_s"]) * effective_sfreq))
    window_samples = int(config["target_window_samples"]) if config.get("target_window_samples") is not None else int(round(float(config["target_window_s"]) * effective_sfreq))
    stop_sample = start_sample + window_samples

    if stop_sample > X_rs.shape[-1]:
        raise ValueError(
            f"Subject {subject_id}: crop [{start_sample}:{stop_sample}] exceeds trial length "
            f"{X_rs.shape[-1]} at effective_sfreq={effective_sfreq}"
        )

    X_win = X_rs[:, :, start_sample:stop_sample]
    runtime_steps.append(f"crop fixed window samples [{start_sample}:{stop_sample}]")

    y = labels_to_zero_based(labels)
    X_win, y, reject_stats = maybe_reject_bad_trials(X_win, y, subject_id, config, runtime_steps)
    preprocessing_stats.update(reject_stats)

    # Save both the intended pipeline and the actual runtime steps.
    preprocessing_stats["pipeline_plan"] = steps
    preprocessing_stats["runtime_steps"] = runtime_steps

    return X_win.astype(np.float32), y.astype(np.int64), int(expected_samples_per_trial), runtime_steps, preprocessing_stats


[2026-06-14 13:45:54] Configured preprocessing pipeline:
[2026-06-14 13:45:54]   - [ON] source::select_eeg_channels | - select Liu EEG channels, drop CPz reference, EOG, and marker before model input
[2026-06-14 13:45:54]   - [off] source::demean | mode=none | baseline_window_s=[0.0, 2.0]
[2026-06-14 13:45:54]   - [off] source::detrend | mode=none
[2026-06-14 13:45:54]   - [off] source::eog_correction | mode=none
[2026-06-14 13:45:54]   - [off] source::artifact_clipping | mode=none | percentile=99.5
[2026-06-14 13:45:54]   - [ON] mne_raw::reference | timing=before_resample_filter | mode=average
[2026-06-14 13:45:54]   - [off] mne_raw::resample | sfreq=500.0
[2026-06-14 13:45:54]   - [ON] mne_raw::bandpass_filter | l_freq=0.5 | h_freq=40.0 | method=fir | phase=zero | fir_design=firwin
[2026-06-14 13:45:54]   - [off] mne_raw::notch_filter | timing=after_bandpass
[2026-06-14 13:45:54]   - [ON] window::crop_fixed_mi_window | start_s=1.5
[2026-06-14 13:45:54]   - [off] window::bad_trial_rej

In [10]:
def find_source_mat_files(root):
    root = Path(root)
    return sorted(root.rglob("*.mat")) if root.exists() else []

def subject_id_from_path(path):
    s = str(path)
    m = re.search(r"sub[-_ ]?(\d{1,2})", s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Could not infer subject id from path: {path}")

def _is_mat_struct(x):
    return hasattr(x, "_fieldnames")

def _walk_mat_object(obj, prefix=""):
    """Recursively walk scipy-loaded MATLAB dicts/structs.

    Liu2024 source files may expose only a top-level `eeg` object instead of
    top-level `rawdata` and `labels`. This walker lets the loader find nested
    arrays without assuming one exact MATLAB struct layout.
    """
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).startswith("__"):
                continue
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif isinstance(obj, np.ndarray):
        if obj.dtype == object and obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                yield from _walk_mat_object(item, f"{prefix}{idx}")

def mat_structure_preview(path, max_rows=200):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    rows = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray):
            rows.append({
                "name": name,
                "type": "ndarray",
                "shape": str(value.shape),
                "dtype": str(value.dtype),
            })
        else:
            rows.append({
                "name": name,
                "type": type(value).__name__,
                "shape": "",
                "dtype": "",
            })
    return pd.DataFrame(rows).head(max_rows)

def _normalize_rawdata_shape(rawdata, labels=None):
    arr = np.asarray(rawdata)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D rawdata, got shape={arr.shape}")

    # Prefer the label-count axis as the trial axis when labels are available.
    if labels is not None:
        n_labels = int(np.asarray(labels).size)
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size == n_labels]
    else:
        trial_axes = []

    if not trial_axes:
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size in (39, 40)]

    if trial_axes and trial_axes[0] != 0:
        arr = np.moveaxis(arr, trial_axes[0], 0)

    # After trial-axis normalization, the time axis should be the largest axis.
    time_axis = int(np.argmax(arr.shape[1:]) + 1)
    if time_axis != 2:
        arr = np.moveaxis(arr, time_axis, 2)

    if arr.shape[1] < 29 or arr.shape[2] < 1000:
        raise ValueError(f"Could not normalize rawdata to trials x channels x samples, got {arr.shape}")
    return arr

def _score_raw_candidate(name, arr):
    lname = name.lower()
    score = 0
    if "rawdata" in lname or "raw" in lname or "data" in lname:
        score += 10
    if arr.ndim == 3:
        score += 5
    if any(size in (39, 40) for size in arr.shape):
        score += 3
    if max(arr.shape) >= 3000:
        score += 2
    if "eeg" in lname:
        score += 1
    return score

def _score_label_candidate(name, arr):
    lname = name.lower()
    flat = np.asarray(arr).ravel()
    unique = set(np.unique(flat).astype(str).tolist()) if flat.size <= 200 else set()
    score = 0
    if "label" in lname or "class" in lname or lname.split(".")[-1] in {"y", "labels"}:
        score += 10
    if flat.size in (39, 40):
        score += 3
    if unique and unique.issubset({"0", "1", "2"}):
        score += 2
    return score

def validate_liu_source_subject(rawdata, labels, subject_id, path=None):
    """Validate the fixed Liu source MAT layout assumptions."""
    expected_trials = LIU_EXPECTED_TRIALS_PER_SUBJECT
    expected_channels = LIU_EXPECTED_SOURCE_CHANNELS
    expected_samples = LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL

    if rawdata.shape[0] != labels.size:
        raise ValueError(
            f"Subject {subject_id}: labels/trials mismatch. "
            f"rawdata={rawdata.shape}, labels={labels.shape}, path={path}"
        )
    if rawdata.shape[0] != expected_trials:
        print(f"WARNING subject {subject_id}: expected {expected_trials} trials, got {rawdata.shape[0]}")
    if rawdata.shape[1] < len(SOURCE_EEG_CHANNEL_INDICES_30):
        raise ValueError(f"Subject {subject_id}: expected at least 30 EEG-like channels, got {rawdata.shape}")
    if rawdata.shape[1] != expected_channels:
        print(f"WARNING subject {subject_id}: expected {expected_channels} source channels, got {rawdata.shape[1]}")
    if rawdata.shape[2] != expected_samples:
        print(f"WARNING subject {subject_id}: expected {expected_samples} samples/trial, got {rawdata.shape[2]}")

    unique = set(np.unique(labels).astype(int).tolist())
    if not unique.issubset({0, 1, 2}):
        raise ValueError(f"Subject {subject_id}: unexpected labels {sorted(unique)}")

    y0 = labels_to_zero_based(labels)
    counts = np.bincount(y0, minlength=TARGET_N_CLASSES)
    if counts.min() == 0:
        raise ValueError(f"Subject {subject_id}: missing class after zero-based conversion, counts={counts.tolist()}")
    if counts[0] != counts[1]:
        print(f"WARNING subject {subject_id}: class counts are not balanced: {counts.tolist()}")

def load_subject_mat(path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)

    arrays = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray) and value.dtype != object:
            arrays.append((name, np.asarray(value)))

    raw_candidates, label_candidates = [], []
    for name, arr in arrays:
        if arr.ndim == 3:
            raw_candidates.append((_score_raw_candidate(name, arr), name, arr))
        elif arr.ndim in (1, 2):
            label_candidates.append((_score_label_candidate(name, arr), name, arr))

    if not raw_candidates or not label_candidates:
        preview = mat_structure_preview(path)
        preview_path = ARTIFACT_DIR / f"mat_structure_failure_{Path(path).stem}.csv"
        preview.to_csv(preview_path, index=False)
        print(f"MAT structure preview for failure saved to: {preview_path}")
        display(preview.head(40))
        raise KeyError(f"Could not locate 3D raw data and labels in {path}")

    _, raw_name, raw_arr = sorted(raw_candidates, key=lambda x: x[0], reverse=True)[0]
    _, label_name, label_arr = sorted(label_candidates, key=lambda x: x[0], reverse=True)[0]

    labels = np.asarray(label_arr).astype(int).ravel()
    rawdata = _normalize_rawdata_shape(raw_arr, labels=labels).astype(np.float64)

    if labels.size != rawdata.shape[0]:
        raise ValueError(f"Label count mismatch in {path}: labels={labels.shape}, rawdata={rawdata.shape}")

    return rawdata, labels.astype(int), raw_name, label_name


In [11]:
class SubjectArrayDataset(Dataset):
    def __init__(self, X, y, subject_id):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)
        self.subject_id = str(subject_id)
        if self.X.ndim != 3:
            raise ValueError(f"SubjectArrayDataset expects X as N x C x T, got shape={self.X.shape}.")
        if len(self.X) != len(self.y):
            raise ValueError(f"X/y length mismatch: {len(self.X)} windows vs {len(self.y)} labels.")

    def __len__(self):
        return int(len(self.y))

    def __getitem__(self, idx):
        x = np.asarray(self.X[idx], dtype=np.float32)
        if x.ndim != 2:
            raise ValueError(f"Expected one EEG window as C x T, got shape={x.shape}.")
        return x, int(self.y[idx])


class FoldNormalizedDataset(Dataset):
    """Wrap a dataset and apply either fold-fitted or trial-wise normalization."""

    def __init__(self, dataset, normalizer_state):
        self.dataset = dataset
        self.normalizer_state = normalizer_state or {"mode": "none"}

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        x = apply_normalizer_to_array(x, self.normalizer_state)
        if x.ndim != 2:
            raise ValueError(f"Normalization must return C x T, got shape={x.shape}.")
        return x.astype(np.float32), int(y)


In [12]:
def _json_safe_float(value, decimals=8):
    if value is None:
        return None
    value = float(np.nan_to_num(value, nan=0.0, posinf=0.0, neginf=0.0))
    return round(value, decimals)

def _json_safe_float_list(values, decimals=8):
    arr = np.asarray(values, dtype=float)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return np.round(arr, decimals=decimals).tolist()


These normalization helpers are carried from the reference's classifier cell because
`FoldNormalizedDataset` (above) calls `apply_normalizer_to_array`. They are fold-safe: the
`train_*` modes are fit on the training split only.

In [13]:
def get_targets(dataset):
    return np.asarray([int(dataset[i][1]) for i in range(len(dataset))], dtype=np.int64)

def dataset_to_array(dataset):
    X = np.stack([np.asarray(dataset[i][0], dtype=np.float32) for i in range(len(dataset))], axis=0)
    if X.ndim != 3:
        raise ValueError(f"Expected dataset windows to stack as N x C x T, got shape={X.shape}.")
    return X

def _safe_scale(scale, eps):
    scale = np.asarray(scale, dtype=np.float32)
    return np.where(np.abs(scale) < float(eps), 1.0, scale).astype(np.float32)

def _as_window_2d(x):
    """Keep each EEG example in C x T format for Braindecode SignalJEPA_PreLocal."""
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 3 and x.shape[0] == 1:
        x = x[0]
    if x.ndim != 2:
        raise ValueError(f"Expected one EEG window to have shape C x T, got shape={x.shape}.")
    return x

def _state_array_for_window(value, x):
    """Convert fitted fold statistics to shapes that broadcast over a single C x T window."""
    arr = np.asarray(value, dtype=np.float32)
    if arr.ndim == 0:
        return arr
    if x.ndim == 2 and arr.ndim == 3 and arr.shape[0] == 1:
        # Old shape from fold fitting was 1 x C x 1. For one example, use C x 1.
        arr = arr[0]
    if x.ndim == 2 and arr.ndim == 1 and arr.shape[0] == x.shape[0]:
        arr = arr[:, None]
    return arr

def apply_normalizer_to_array(x, state):
    """Apply a fitted fold normalizer or trial-wise normalizer to one C x T example.

    Important: this function must return C x T, not 1 x C x T.
    Returning 1 x C x T makes the DataLoader batch 4D and breaks SignalJEPA_PreLocal.
    """
    mode = str((state or {}).get("mode", "none")).lower()
    eps = float((state or {}).get("eps", CONFIG.get("normalization_eps", 1e-6)))
    x = _as_window_2d(x)

    if mode in ("none", "off", "false"):
        return x

    if mode == "train_global_zscore":
        mean = _state_array_for_window(state["mean"], x)
        scale = _state_array_for_window(state["scale"], x)
        return _as_window_2d((x - mean) / scale)

    if mode == "train_channel_zscore":
        mean = _state_array_for_window(state["mean"], x)
        scale = _state_array_for_window(state["scale"], x)
        return _as_window_2d((x - mean) / scale)

    if mode == "train_channel_robust":
        median = _state_array_for_window(state["median"], x)
        scale = _state_array_for_window(state["scale"], x)
        return _as_window_2d((x - median) / scale)

    if mode == "trial_global_zscore":
        mean = x.mean(keepdims=True)
        scale = max(float(x.std()), eps)
        return _as_window_2d((x - mean) / scale)

    if mode == "trial_channel_zscore":
        mean = x.mean(axis=-1, keepdims=True)
        scale = _safe_scale(x.std(axis=-1, keepdims=True), eps)
        return _as_window_2d((x - mean) / scale)

    raise ValueError(f"Unsupported normalization_mode={mode}")

def fit_fold_normalizer(train_dataset):
    """Fit normalization on the training fold only where applicable."""
    mode = str(CONFIG.get("normalization_mode", "none")).lower()
    eps = float(CONFIG.get("normalization_eps", 1e-6))
    state = {"mode": mode, "eps": eps}
    summary = {"mode": mode, "fit_scope": "none"}

    if mode in ("none", "off", "false", "trial_global_zscore", "trial_channel_zscore"):
        if mode.startswith("trial_"):
            summary["fit_scope"] = "trial_only_no_fold_fit"
        return state, summary

    X = dataset_to_array(train_dataset)  # N x C x T

    if mode == "train_global_zscore":
        mean = np.asarray(X.mean(), dtype=np.float32)
        scale = np.asarray(max(float(X.std()), eps), dtype=np.float32)
        state.update({"mean": mean, "scale": scale})
        summary.update({
            "fit_scope": "training_fold",
            "mean_shape": [],
            "scale_shape": [],
            "train_mean": float(mean),
            "train_scale": float(scale),
        })
        return state, summary

    if mode == "train_channel_zscore":
        # Shape is C x 1 so it broadcasts correctly over one C x T window.
        mean = X.mean(axis=(0, 2)).astype(np.float32)[:, None]
        scale = _safe_scale(X.std(axis=(0, 2)).astype(np.float32)[:, None], eps)
        state.update({"mean": mean, "scale": scale})
        summary.update({
            "fit_scope": "training_fold",
            "mean_shape": list(mean.shape),
            "scale_shape": list(scale.shape),
            "mean_scale_min": float(scale.min()),
            "mean_scale_max": float(scale.max()),
        })
        return state, summary

    if mode == "train_channel_robust":
        # Shape is C x 1 so it broadcasts correctly over one C x T window.
        median = np.median(X, axis=(0, 2)).astype(np.float32)[:, None]
        q75 = np.percentile(X, 75, axis=(0, 2)).astype(np.float32)[:, None]
        q25 = np.percentile(X, 25, axis=(0, 2)).astype(np.float32)[:, None]
        iqr = _safe_scale((q75 - q25).astype(np.float32), eps)
        state.update({"median": median, "scale": iqr})
        summary.update({
            "fit_scope": "training_fold",
            "median_shape": list(median.shape),
            "scale_shape": list(iqr.shape),
            "iqr_min": float(iqr.min()),
            "iqr_max": float(iqr.max()),
        })
        return state, summary

    raise ValueError(f"Unsupported normalization_mode={CONFIG.get('normalization_mode')}")

def maybe_wrap_normalized(train_set, test_set):
    state, summary = fit_fold_normalizer(train_set)
    mode = str(summary.get("mode", "none")).lower()
    if mode in ("none", "off", "false"):
        return train_set, test_set, summary
    return FoldNormalizedDataset(train_set, state), FoldNormalizedDataset(test_set, state), summary


## 3.3 Pipeline-plan trace — where does resampling sit?

Builds the human-readable plan for the active config and pinpoints the resample step's
position relative to referencing, filtering, and windowing.

In [14]:
PREPROCESSING_PIPELINE = build_preprocessing_pipeline(CONFIG)
print("Configured preprocessing pipeline (active sfreq_mode = "
      f"{CONFIG['sfreq_mode']}, effective {EFFECTIVE_SFREQ:g} Hz):")
plan_lines = describe_pipeline(PREPROCESSING_PIPELINE)
for i, line in enumerate(plan_lines):
    print(f"  {i:2d}. {line}")

# Locate the resample + window steps in the plan.
def _find_step(pipeline, name):
    for pos, step in enumerate(pipeline):
        if step.get("name") == name:
            return pos, step
    return None, None

resample_pos, resample_step = _find_step(PREPROCESSING_PIPELINE, "resample")
filter_pos, _ = _find_step(PREPROCESSING_PIPELINE, "bandpass_filter")
window_pos, _ = _find_step(PREPROCESSING_PIPELINE, "crop_fixed_mi_window")

print("\nResample-step trace:")
print(f"  resample step index:   {resample_pos}")
print(f"  resample enabled:      {resample_step.get('enabled')}")
print(f"  resample target sfreq: {resample_step.get('sfreq')}")
print(f"  bandpass step index:   {filter_pos}  (resample {'before' if resample_pos < filter_pos else 'after'} filter)")
print(f"  window-crop step index:{window_pos}  (crop happens at effective sfreq = {EFFECTIVE_SFREQ:g} Hz)")
with open(ARTIFACT_DIR / "preprocessing_plan.json", "w") as f:
    json.dump(PREPROCESSING_PIPELINE, f, indent=2, default=str)


[2026-06-14 13:45:54] Configured preprocessing pipeline (active sfreq_mode = native_500, effective 500 Hz):
[2026-06-14 13:45:54]    0. [ON] source::select_eeg_channels | - select Liu EEG channels, drop CPz reference, EOG, and marker before model input
[2026-06-14 13:45:54]    1. [off] source::demean | mode=none | baseline_window_s=[0.0, 2.0]
[2026-06-14 13:45:54]    2. [off] source::detrend | mode=none
[2026-06-14 13:45:54]    3. [off] source::eog_correction | mode=none
[2026-06-14 13:45:54]    4. [off] source::artifact_clipping | mode=none | percentile=99.5
[2026-06-14 13:45:54]    5. [ON] mne_raw::reference | timing=before_resample_filter | mode=average
[2026-06-14 13:45:54]    6. [off] mne_raw::resample | sfreq=500.0
[2026-06-14 13:45:54]    7. [ON] mne_raw::bandpass_filter | l_freq=0.5 | h_freq=40.0 | method=fir | phase=zero | fir_design=firwin
[2026-06-14 13:45:54]    8. [off] mne_raw::notch_filter | timing=after_bandpass
[2026-06-14 13:45:54]    9. [ON] window::crop_fixed_mi_win

## 3.4 Empirical shape audit on one trial

Confirms the static audit against the *actual* preprocessing function: trial shapes before
and after, the sampling rate MNE actually produces, samples per trial after resampling, and
the crop indices in both samples and seconds. Uses a real subject if `.mat` files are found,
otherwise a clearly-labelled synthetic `40 x 33 x 4000` trial at 500 Hz so the audit still
runs. Requires MNE.

In [15]:
def _make_synthetic_subject(seed=2026):
    """Synthetic Liu-shaped source trial: 40 trials x 33 channels x 4000 samples @500 Hz."""
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((LIU_EXPECTED_TRIALS_PER_SUBJECT,
                             LIU_EXPECTED_SOURCE_CHANNELS,
                             LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL)).astype(np.float64) * 10.0
    labels = np.array([1, 2] * (LIU_EXPECTED_TRIALS_PER_SUBJECT // 2), dtype=int)
    return X, labels

empirical_audit = {"ran": False}
if not HAVE_MNE:
    print("[3.4] MNE unavailable -> skipping empirical shape audit. "
          "Static audit (3.1) and plan trace (3.3) still hold.")
else:
    mat_files = find_source_mat_files(SOURCE_EXTRACT_DIR)
    if mat_files:
        sid = subject_id_from_path(mat_files[0])
        X_raw, y_raw, raw_field, label_field = load_subject_mat(mat_files[0])
        source_label = f"real subject {sid} ({Path(mat_files[0]).name})"
    elif CONFIG.get("audit_synthetic_if_missing", True):
        sid = 0
        X_raw, y_raw = _make_synthetic_subject()
        raw_field, label_field = "synthetic", "synthetic"
        source_label = "SYNTHETIC 40x33x4000 @500 Hz (no .mat files found)"
    else:
        X_raw = None
        print("[3.4] No .mat files and synthetic fallback disabled -> skipping.")

    if X_raw is not None:
        before_shape = tuple(np.asarray(X_raw).shape)
        X_win, y, samples_per_trial, steps, stats = preprocess_subject_configurable(X_raw, y_raw, sid)
        after_shape = tuple(X_win.shape)
        realized_sfreq = round(samples_per_trial * LIU_SOURCE_SFREQ / LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL, 4)

        empirical_audit = {
            "ran": True,
            "source": source_label,
            "raw_field": raw_field,
            "label_field": label_field,
            "source_sfreq_hz": float(LIU_SOURCE_SFREQ),
            "configured_effective_sfreq_hz": float(EFFECTIVE_SFREQ),
            "realized_effective_sfreq_hz": float(realized_sfreq),
            "raw_shape_trials_chans_samples": list(before_shape),
            "preprocessed_shape_trials_chans_samples": list(after_shape),
            "resampled_samples_per_trial": int(samples_per_trial),
            "window_samples": int(WINDOW_SAMPLES),
            "crop_start_sample": int(MI_WINDOW_START_SAMPLE),
            "crop_stop_sample": int(MI_WINDOW_STOP_SAMPLE),
            "crop_start_s": round(MI_WINDOW_START_SAMPLE / EFFECTIVE_SFREQ, 6),
            "crop_stop_s": round(MI_WINDOW_STOP_SAMPLE / EFFECTIVE_SFREQ, 6),
            "window_duration_s": float(TARGET_TRIAL_DURATION_S),
            "class_counts": np.bincount(y, minlength=TARGET_N_CLASSES).tolist(),
            "runtime_steps": steps,
        }
        print("=" * 78)
        print("EMPIRICAL SHAPE AUDIT")
        print("=" * 78)
        print(f"  Source:                 {source_label}")
        print(f"  Raw shape (T x C x S):  {before_shape}")
        print(f"  After preprocessing:    {after_shape}  (T x C x window_samples)")
        print(f"  Configured eff. sfreq:  {EFFECTIVE_SFREQ:g} Hz")
        print(f"  Realized eff. sfreq:    {realized_sfreq:g} Hz  "
              f"({'MATCH' if abs(realized_sfreq - EFFECTIVE_SFREQ) < 1e-6 else 'MISMATCH - investigate'})")
        print(f"  Resampled samples/trial:{samples_per_trial}")
        print(f"  Crop: samples [{MI_WINDOW_START_SAMPLE}:{MI_WINDOW_STOP_SAMPLE}] "
              f"= seconds [{empirical_audit['crop_start_s']:.4f}:{empirical_audit['crop_stop_s']:.4f}]")
        print("\n  Realized runtime steps:")
        for s in steps:
            print(f"    - {s}")
        with open(ARTIFACT_DIR / "sfreq_audit_empirical.json", "w") as f:
            json.dump(empirical_audit, f, indent=2, default=str)
        print(f"\n  Saved: {ARTIFACT_DIR / 'sfreq_audit_empirical.json'}")


[2026-06-14 13:45:54] ==============================================================================
[2026-06-14 13:45:54] EMPIRICAL SHAPE AUDIT
[2026-06-14 13:45:54] ==============================================================================
[2026-06-14 13:45:54]   Source:                 real subject 1 (sub-01_task-motor-imagery_eeg.mat)
[2026-06-14 13:45:54]   Raw shape (T x C x S):  (40, 33, 4000)
[2026-06-14 13:45:54]   After preprocessing:    (40, 29, 2100)  (T x C x window_samples)
[2026-06-14 13:45:54]   Configured eff. sfreq:  500 Hz
[2026-06-14 13:45:54]   Realized eff. sfreq:    500 Hz  (MATCH)
[2026-06-14 13:45:54]   Resampled samples/trial:4000
[2026-06-14 13:45:54]   Crop: samples [750:2850] = seconds [1.5000:5.7000]

[2026-06-14 13:45:54]   Realized runtime steps:
[2026-06-14 13:45:54]     - select 29 EEG channels; drop CPz source reference, EOG, and marker
[2026-06-14 13:45:54]     - source demean skipped
[2026-06-14 13:45:54]     - source detrend skipped
[2026-06-14

# 4. Native-500 Hz path & architecture compatibility

The S-JEPA local encoder is a stack of temporal convolutions whose kernels are defined in
**samples**, not seconds. So the same architecture sees a different real-time receptive
field and produces a different number of latent time-steps ("tokens") at 128 Hz vs 500 Hz.
This section makes that concrete by building the model at the configured rate (4.1) and
diagnosing both rates with a dummy forward pass (4.2).

## 4.1 Build the model at the configured sampling rate

`build_model` is carried verbatim; `n_times = WINDOW_SAMPLES` is now whatever the
seconds-first window resolves to at the active rate.

In [16]:
CH_NAMES = list(EEG_CHANNEL_NAMES)
CHS_INFO = make_liu_info(EFFECTIVE_SFREQ)["chs"] if HAVE_MNE else None
if CHS_INFO is None:
    print("[4.1] MNE unavailable -> CHS_INFO=None; the 'without-chans' S-JEPA variant does not require it.")


In [17]:
NEW_LAYER_PREFIXES = ("spatial_conv.", "final_layer.")

def build_model():
    common_kwargs = {
        "n_chans": len(CH_NAMES),
        "chs_info": CHS_INFO,
        "n_times": WINDOW_SAMPLES,
        "n_outputs": TARGET_N_CLASSES,
    }
    mode = CONFIG["pretrained_mode"]
    if mode == "from_pretrained":
        model = SignalJEPA_PreLocal.from_pretrained(
            CONFIG["pretrained_repo_id"],
            **common_kwargs,
            strict=False,
        )
        info = {
            "loading_path": "from_pretrained",
            "repo_id": CONFIG["pretrained_repo_id"],
            "mode": mode,
        }
    elif mode == "random":
        model = SignalJEPA_PreLocal(**common_kwargs)
        info = {
            "loading_path": "random_initialization",
            "repo_id": None,
            "mode": mode,
        }
    else:
        raise ValueError("pretrained_mode must be 'from_pretrained' or 'random'.")
    info["model_name"] = CONFIG["model_name"]
    return model, info

def set_trainable_params_for_phase(model, phase):
    if phase not in ("new", "warmup", "full"):
        raise ValueError(f"Unsupported phase: {phase}")

    if phase == "full":
        for _, p in model.named_parameters():
            p.requires_grad = True
        phase_groups = ["all_parameters"]
    else:
        for _, p in model.named_parameters():
            p.requires_grad = False
        for name, p in model.named_parameters():
            if any(name.startswith(prefix) for prefix in NEW_LAYER_PREFIXES):
                p.requires_grad = True
        phase_groups = list(NEW_LAYER_PREFIXES)

    trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if trainable == 0:
        raise RuntimeError(f"No trainable parameters for phase={phase}.")
    return {
        "phase": phase,
        "trainable_groups": phase_groups,
        "total_params": int(total),
        "trainable_params": int(trainable),
        "trainable_ratio": float(trainable / total),
        "trainable_names": trainable_names,
    }

def summarize_trainable_parameters(model):
    rows = []
    for name, param in model.named_parameters():
        if param.requires_grad:
            rows.append({
                "name": name,
                "numel": int(param.numel()),
                "shape": list(param.shape),
            })
    return rows

def count_trainable_from_rows(rows):
    return int(sum(row.get("numel", 0) for row in rows))


In [18]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return int(total), int(trainable)

ACTIVE_MODEL_INFO = {"built": False}
if not HAVE_BRAINDECODE or not HAVE_TORCH:
    print("[4.1] torch/braindecode unavailable -> skipping model build.")
else:
    try:
        _model, _pre_info = build_model()
        _phase = set_trainable_params_for_phase(_model, CONFIG["strategy"])
        total, trainable = count_params(_model)
        ACTIVE_MODEL_INFO = {
            "built": True,
            "model_name": CONFIG["model_name"],
            "pretrained_mode": CONFIG["pretrained_mode"],
            "effective_sfreq_hz": float(EFFECTIVE_SFREQ),
            "n_chans": len(CH_NAMES),
            "n_times": int(WINDOW_SAMPLES),
            "n_outputs": TARGET_N_CLASSES,
            "total_params": total,
            "trainable_params_for_strategy": _phase["trainable_params"],
            "trainable_groups": _phase["trainable_groups"],
            "load_info": _pre_info,
        }
        print(f"Model built at {EFFECTIVE_SFREQ:g} Hz | n_times={WINDOW_SAMPLES} | "
              f"params total={total:,} | trainable({CONFIG['strategy']})={_phase['trainable_params']:,}")
    except Exception as exc:
        ACTIVE_MODEL_INFO = {"built": False, "error": str(exc)}
        print(f"[4.1] Model build failed (often a network/checkpoint download issue): {exc}")


[2026-06-14 13:45:55] Model built at 500 Hz | n_times=2100 | params total=22,154 | trainable(new)=8,314


## 4.2 Architecture-compatibility diagnostics: 128 Hz vs 500 Hz

Builds the model at both window lengths, runs a dummy forward pass with hooks to measure the
latent time-step (token) sequence length, extracts the temporal-conv kernel sizes to report
the receptive field **in seconds**, and records parameter counts. This exposes the real
architectural consequences of the rate change rather than hacking tensor shapes.

> **Caveat — checkpoint sampling rate.** `from_pretrained` loads braindecode's S-JEPA
> checkpoint, whose conv kernels were learned at whatever sampling rate it was pretrained on.
> Applying those fixed-in-samples kernels at a different rate changes their effective
> frequency response. Verify the checkpoint's pretraining sfreq from its model card before
> drawing conclusions; the diagnostics below print any sfreq-like attribute found on the
> loaded model.

In [19]:
def _temporal_conv_kernel_report(model, effective_sfreq):
    rows = []
    for name, mod in model.named_modules():
        w = getattr(mod, "weight", None)
        ks = getattr(mod, "kernel_size", None)
        if w is not None and ks is not None and hasattr(w, "ndim") and w.ndim >= 3:
            ks_tuple = ks if isinstance(ks, (tuple, list)) else (ks,)
            ks_time = int(ks_tuple[-1])
            rows.append({
                "module": name,
                "type": type(mod).__name__,
                "kernel_time_samples": ks_time,
                "kernel_time_s": round(ks_time / effective_sfreq, 5),
                "out_channels": int(getattr(mod, "out_channels", -1)),
            })
    return rows

def _measure_forward(model, n_chans, n_times):
    """Run a dummy batch and capture leaf-module output shapes + final output shape."""
    model.eval()
    leaf_shapes = {}
    handles = []
    def mk(nm):
        def hook(m, inp, out):
            try:
                if hasattr(out, "shape"):
                    leaf_shapes[nm] = tuple(int(s) for s in out.shape)
            except Exception:
                pass
        return hook
    for nm, mod in model.named_modules():
        if len(list(mod.children())) == 0:
            handles.append(mod.register_forward_hook(mk(nm)))
    out_shape, err = None, None
    for shaper in (lambda: torch.zeros(2, n_chans, n_times),
                   lambda: torch.zeros(2, 1, n_chans, n_times)):
        try:
            with torch.no_grad():
                out = model(shaper())
            out_shape = tuple(int(s) for s in out.shape) if hasattr(out, "shape") else None
            err = None
            break
        except Exception as exc:
            err = str(exc)
            continue
    for h in handles:
        h.remove()
    # Heuristic token length: the smallest >1 trailing time-dim among 3D leaf outputs.
    time_dims = sorted({sh[-1] for sh in leaf_shapes.values() if len(sh) >= 3 and sh[-1] > 1})
    return {"output_shape": out_shape, "forward_error": err,
            "observed_trailing_time_dims": time_dims, "n_leaf_modules": len(leaf_shapes)}

ARCH_COMPARISON = {"ran": False}
if not HAVE_BRAINDECODE or not HAVE_TORCH:
    print("[4.2] torch/braindecode unavailable -> skipping architecture diagnostics.")
else:
    seed_everything(int(CONFIG["seed"]))
    rows = []
    for mode in CONFIG.get("compare_sfreq_modes", ["resample_128", "native_500"]):
        info = resolve_sampling_and_window({**CONFIG, "sfreq_mode": mode})
        n_times = int(info["window_samples"])
        eff = info["effective_sfreq"]
        try:
            # Build at this n_times. Param COUNTS are init-independent, so fall back to a
            # random init if the pretrained checkpoint can't be fetched (e.g. offline).
            common = {"n_chans": len(CH_NAMES), "chs_info": CHS_INFO,
                      "n_times": n_times, "n_outputs": TARGET_N_CLASSES}
            try:
                m = SignalJEPA_PreLocal.from_pretrained(CONFIG["pretrained_repo_id"], **common, strict=False)
                init = "from_pretrained"
            except Exception as exc_pt:
                m = SignalJEPA_PreLocal(**common)
                init = f"random_fallback ({type(exc_pt).__name__})"
            total, _ = count_params(m)
            fwd = _measure_forward(m, len(CH_NAMES), n_times)
            kernels = _temporal_conv_kernel_report(m, eff)
            first_k = kernels[0] if kernels else {}
            sfreq_attrs = {a: getattr(m, a) for a in ("sfreq", "fs", "sampling_rate")
                           if hasattr(m, a)}
            rows.append({
                "sfreq_mode": mode,
                "effective_sfreq_hz": eff,
                "n_times": n_times,
                "init": init,
                "total_params": total,
                "output_shape": fwd["output_shape"],
                "observed_trailing_time_dims": fwd["observed_trailing_time_dims"],
                "first_temporal_kernel_samples": first_k.get("kernel_time_samples"),
                "first_temporal_kernel_s": first_k.get("kernel_time_s"),
                "model_sfreq_attrs": sfreq_attrs,
                "forward_error": fwd["forward_error"],
                "kernels": kernels,
            })
            print(f"  [{mode}] n_times={n_times} @ {eff:g} Hz | params={total:,} | "
                  f"out={fwd['output_shape']} | token_time_dims={fwd['observed_trailing_time_dims']} | "
                  f"first_kernel={first_k.get('kernel_time_samples')} samp "
                  f"({first_k.get('kernel_time_s')} s) | init={init}")
        except Exception as exc:
            rows.append({"sfreq_mode": mode, "error": str(exc)})
            print(f"  [{mode}] architecture probe failed: {exc}")
    ARCH_COMPARISON = {"ran": True, "modes": rows}
    # Compact comparison table (drop the verbose per-kernel list).
    arch_df = pd.DataFrame([{k: v for k, v in r.items() if k != "kernels"} for r in rows])
    try:
        from IPython.display import display
        display(arch_df)
    except Exception:
        print(arch_df.to_string(index=False))
    with open(ARTIFACT_DIR / "architecture_compatibility.json", "w") as f:
        json.dump(ARCH_COMPARISON, f, indent=2, default=str)
    print(f"\n  Saved: {ARTIFACT_DIR / 'architecture_compatibility.json'}")
    print("\n  Interpretation: param count typically does NOT change with n_times (convs are "
          "weight-shared over time), but the token/time-step sequence length DOES grow with "
          "more samples, so the encoder processes a longer latent sequence at 500 Hz and each "
          "fixed-sample kernel covers less real time.")


[2026-06-14 13:45:56]   [resample_128] n_times=538 @ 128 Hz | params=16,010 | out=(2, 2) | token_time_dims=[4, 8, 16, 32, 64, 538] | first_kernel=32 samp (0.25 s) | init=from_pretrained
[2026-06-14 13:45:56]   [native_500] n_times=2100 @ 500 Hz | params=22,154 | out=(2, 2) | token_time_dims=[16, 32, 64, 129, 259, 2100] | first_kernel=32 samp (0.064 s) | init=from_pretrained


,sfreq_mode,effective_sfreq_hz,n_times,init,total_params,output_shape,observed_trailing_time_dims,first_temporal_kernel_samples,first_temporal_kernel_s,model_sfreq_attrs,forward_error
0,resample_128,128.0,538,from_pretrained,16010,"(2, 2)","[4, 8, 16, 32, 64, 538]",32,0.250,{'sfreq': 128.0},None
1,native_500,500.0,2100,from_pretrained,22154,"(2, 2)","[16, 32, 64, 129, 259, 2100]",32,0.064,{'sfreq': 128.0},None



[2026-06-14 13:45:56]   Saved: /home/vegorov/Repos/eeg_jepa_research/artifacts/liu2024-source-mat-sjepa-pretrain-500hz-audit/20260614_1345_native500_sr500_efdd1f75/architecture_compatibility.json

[2026-06-14 13:45:56]   Interpretation: param count typically does NOT change with n_times (convs are weight-shared over time), but the token/time-step sequence length DOES grow with more samples, so the encoder processes a longer latent sequence at 500 Hz and each fixed-sample kernel covers less real time.


# 5. Fair 128-vs-500 comparison

Two pieces: (5.1) generate drop-in config files for each rate so the *exact same* reference
training pipeline produces both runs into separate folders, and (5.2) a reader that tabulates
the two runs' saved metrics once they exist.

## 5.1 Generate drop-in configs for each sampling rate

These configs differ **only** in `sfreq_mode`/`run_tag` (and the derived resample fields).
Everything else — channels, filter, window seconds, evaluation protocol, seeds, model — is
held fixed, which is what keeps the comparison fair. Run each in the reference notebook
(`liu2024_source_mat_sjepa_prelocal_augmented_clean.ipynb`) by loading the file in place of
its inline `CONFIG`, or flip this notebook's own `sfreq_mode` and re-run with the full
training pipeline.

In [20]:
def make_sfreq_variant_config(base_config, sfreq_mode, run_tag=None):
    cfg = copy.deepcopy(base_config)
    cfg["sfreq_mode"] = sfreq_mode
    cfg["run_tag"] = run_tag or sfreq_mode
    cfg["target_window_samples"] = None          # always seconds-first
    sr = resolve_sfreq_mode(cfg)
    cfg["resample"] = sr["resample"]
    cfg["resample_sfreq"] = sr["resample_sfreq"]
    cfg["effective_sfreq"] = sr["effective_sfreq"]
    cfg["sfreq"] = sr["effective_sfreq"]
    return cfg

variant_paths = {}
for mode in CONFIG.get("compare_sfreq_modes", ["resample_128", "native_500"]):
    cfg = make_sfreq_variant_config(CONFIG, mode)
    win = resolve_sampling_and_window(cfg)
    fname = f"config_{int(round(win['effective_sfreq']))}hz.json"
    path = ARTIFACT_DIR / fname
    with open(path, "w") as f:
        json.dump(cfg, f, indent=2, default=str)
    variant_paths[mode] = str(path)
    print(f"  {mode:14s}-> {fname}: effective {win['effective_sfreq']:g} Hz | "
          f"window {win['window_samples']} samples = {win['effective_window_duration_s']:.4f} s")

print("\nDrop-in usage: load one of these into the reference notebook's CONFIG cell. Held "
      "fixed across both: channels, 0.5-40 Hz bandpass, 4.2 s window (seconds-first), "
      "stratified within-subject 5-fold CV, seeds, and SignalJEPA_PreLocal.")


[2026-06-14 13:45:56]   resample_128  -> config_128hz.json: effective 128 Hz | window 538 samples = 4.2031 s
[2026-06-14 13:45:56]   native_500    -> config_500hz.json: effective 500 Hz | window 2100 samples = 4.2000 s

[2026-06-14 13:45:56] Drop-in usage: load one of these into the reference notebook's CONFIG cell. Held fixed across both: channels, 0.5-40 Hz bandpass, 4.2 s window (seconds-first), stratified within-subject 5-fold CV, seeds, and SignalJEPA_PreLocal.


## 5.2 Compare two completed runs

Once you have run both rates, point this at the two run folders to read their
`global_metrics.json` and tabulate accuracy / balanced accuracy side by side. Until both
exist it prints instructions rather than failing.

In [21]:
def load_run_summary(run_dir):
    run_dir = Path(run_dir)
    out = {"run_dir": str(run_dir)}
    for fname, key in [("global_metrics.json", "global_metrics"),
                       ("run_metadata.json", "run_metadata"),
                       ("config.json", "config")]:
        p = run_dir / fname
        if p.exists():
            try:
                out[key] = json.load(open(p))
            except Exception as exc:
                out[key] = {"_read_error": str(exc)}
    return out

def compare_runs(run_dir_128, run_dir_500):
    rows = []
    for label, rd in [("128 Hz", run_dir_128), ("500 Hz", run_dir_500)]:
        s = load_run_summary(rd)
        gm = s.get("global_metrics", {}) or {}
        meta = s.get("run_metadata", {}) or {}
        rows.append({
            "rate": label,
            "run_dir": s["run_dir"],
            "effective_sfreq": meta.get("effective_sfreq"),
            "window_samples": meta.get("window_samples"),
            "mean_accuracy": gm.get("mean_accuracy"),
            "std_accuracy": gm.get("std_accuracy"),
            "mean_balanced_accuracy": gm.get("mean_balanced_accuracy"),
            "std_balanced_accuracy": gm.get("std_balanced_accuracy"),
            "n_subjects": gm.get("n_subjects"),
            "n_folds_total": gm.get("n_folds_total"),
        })
    df = pd.DataFrame(rows)
    cmp_path = ARTIFACT_DIR / "sfreq_comparison_128_vs_500.csv"
    df.to_csv(cmp_path, index=False)
    print(df.to_string(index=False))
    print(f"\nSaved: {cmp_path}")
    if HAVE_MPL and df["mean_balanced_accuracy"].notna().all():
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.bar(df["rate"], df["mean_balanced_accuracy"],
               yerr=df["std_balanced_accuracy"], capsize=6)
        ax.axhline(0.5, ls="--", c="grey", lw=1, label="chance")
        ax.set_ylabel("mean balanced accuracy"); ax.set_ylim(0, 1)
        ax.set_title("128 Hz vs 500 Hz (within-subject CV)"); ax.legend()
        plot_path = ARTIFACT_DIR / "sfreq_comparison_128_vs_500.png"
        fig.tight_layout(); fig.savefig(plot_path, dpi=160); plt.close(fig)
        print(f"Saved plot: {plot_path}")
    return df

# Example (edit to your actual completed run folders, then uncomment):
# RUN_128 = "/path/to/artifacts/.../<128hz run id>"
# RUN_500 = "/path/to/artifacts/.../<500hz run id>"
# compare_runs(RUN_128, RUN_500)
print("compare_runs(run_dir_128, run_dir_500) is ready. After running both rates "
      "(via the configs from 5.1), set the two run folders and call it.")


[2026-06-14 13:45:56] compare_runs(run_dir_128, run_dir_500) is ready. After running both rates (via the configs from 5.1), set the two run folders and call it.


## 5.3 Save audit run metadata

In [22]:
run_metadata = {
    "run_id": RUN_ID,
    "artifact_dir": str(ARTIFACT_DIR),
    "experiment_name": CONFIG.get("experiment_name"),
    "config_note": CONFIG.get("config_note"),
    "purpose": "sampling-rate audit + sfreq-agnostic 500 Hz enablement (no full CV run here)",
    "source_sfreq": LIU_SOURCE_SFREQ,
    "active_sfreq_mode": CONFIG["sfreq_mode"],
    "effective_sfreq": EFFECTIVE_SFREQ,
    "window_samples": WINDOW_SAMPLES,
    "mi_window_start_s": CONFIG["mi_window_start_s"],
    "mi_window_start_sample": MI_WINDOW_START_SAMPLE,
    "mi_window_stop_sample": MI_WINDOW_STOP_SAMPLE,
    "target_window_duration_s": TARGET_TRIAL_DURATION_S,
    "target_window_samples_config": CONFIG.get("target_window_samples"),
    "channel_names": list(CH_NAMES),
    "n_channels": len(CH_NAMES),
    "preprocessing_config": PREPROCESSING_CONFIG,
    "evaluation_config": EVALUATION_CONFIG,
    "model_name": CONFIG["model_name"],
    "pretrained_mode": CONFIG["pretrained_mode"],
    "pretrained_repo_id": CONFIG["pretrained_repo_id"],
    "static_audit": audit_full if "audit_full" in globals() else None,
    "empirical_audit_ran": bool(empirical_audit.get("ran", False)) if "empirical_audit" in globals() else False,
    "architecture_comparison_ran": bool(ARCH_COMPARISON.get("ran", False)) if "ARCH_COMPARISON" in globals() else False,
    "active_model_info": ACTIVE_MODEL_INFO if "ACTIVE_MODEL_INFO" in globals() else None,
    "variant_config_paths": variant_paths if "variant_paths" in globals() else None,
    "seed": int(CONFIG["seed"]),
}
run_metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(run_metadata_path, "w") as f:
    json.dump(run_metadata, f, indent=2, default=str)
print(f"Audit run metadata saved to: {run_metadata_path}")
print(f"\nAll audit artifacts in: {ARTIFACT_DIR}")
for p in sorted(ARTIFACT_DIR.glob("*")):
    print(f"  - {p.name}")
try:
    _LOG_FILE_HANDLE.close()
except Exception:
    pass


[2026-06-14 13:45:56] Audit run metadata saved to: /home/vegorov/Repos/eeg_jepa_research/artifacts/liu2024-source-mat-sjepa-pretrain-500hz-audit/20260614_1345_native500_sr500_efdd1f75/run_metadata.json

[2026-06-14 13:45:56] All audit artifacts in: /home/vegorov/Repos/eeg_jepa_research/artifacts/liu2024-source-mat-sjepa-pretrain-500hz-audit/20260614_1345_native500_sr500_efdd1f75
[2026-06-14 13:45:56]   - architecture_compatibility.json
[2026-06-14 13:45:56]   - config.json
[2026-06-14 13:45:56]   - config_128hz.json
[2026-06-14 13:45:56]   - config_500hz.json
[2026-06-14 13:45:56]   - preprocessing_plan.json
[2026-06-14 13:45:56]   - run.log
[2026-06-14 13:45:56]   - run_metadata.json
[2026-06-14 13:45:56]   - sfreq_audit_empirical.json
[2026-06-14 13:45:56]   - sfreq_audit_static.csv
[2026-06-14 13:45:56]   - sfreq_audit_static.json


# 6. Summary — what changed, what stayed comparable

**What changed (the fix).**
- The window is now defined **in seconds first** (`target_window_samples=None`), so its
  *duration* is invariant to the sampling rate. The reference's pinned `537` would have
  silently shrunk the window from 4.2 s to ~1.07 s at 500 Hz; the audit makes that explicit.
- A single `sfreq_mode` switch (`native_500` / `resample_128` / `custom`) controls
  resampling coherently, removing scattered hard-coded 128 Hz assumptions. (One more sample-
  count hazard is flagged but not auto-fixed: `smooth_time_mask`'s `mask_len_samples` — hence
  augmentation is off by default in this audit.)
- Run folders are tagged with the effective sampling rate so 128 Hz and 500 Hz runs never
  overwrite each other.

**What stayed comparable (held fixed across rates).**
Channel set and order (29, CPz dropped), 0.5–40 Hz FIR bandpass, average reference timing,
the 4.2 s window starting at 1.5 s, fold-safe normalization, the within-subject stratified
5-fold protocol, all seeds, and the `SignalJEPA_PreLocal` model. The preprocessing,
data-loading, dataset, and model-build code is the reference notebook's, verbatim — so a run
from `config_500hz.json` is directly diff-comparable to a reference 128 Hz run.

**Scientific caveats to raise with your advisor.**
1. *No accuracy claim.* Higher rate ≠ better. With a 40 Hz low-pass, 128 Hz already
   satisfies Nyquist for the retained band; 500 Hz mostly adds samples in an already-filtered
   range. The audit measures the trade-off; it does not assert a winner.
2. *Pretrained-checkpoint sampling rate.* `from_pretrained` kernels are fixed in samples.
   Running them at a rate different from their pretraining rate changes their effective
   frequency response — verify the checkpoint's sfreq before interpreting 500 Hz results.
3. *Token-sequence length.* The encoder produces more latent time-steps at 500 Hz
   (Section 4.2), increasing compute/memory and the effective sequence the model must model;
   with only 40 trials/subject this can hurt as easily as help.
4. *This is transfer + fine-tuning, not SSL pretraining on Liu.* If the goal is genuine
   S-JEPA self-supervised pretraining on Liu 500 Hz, that is a separate notebook to scope.